# 02 - Train LSTM, evaluate on the FD001 test set

2-layer LSTM (hidden 64, dropout 0.3) + small MLP head, Adam, MSE, early stopping on **validation RMSE**, CPU / CUDA / Apple-MPS auto-detected.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
%matplotlib inline
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.config import Config, ARTIFACT_DIR
from src.train import train, predict_cycles
from src.evaluate import metrics
from src import plots
cfg = Config()          # edit hyper-parameters here, e.g. Config(hidden_size=128, device="mps")
cfg.to_dict()

{'subset': 'FD001',
 'rul_cap': 125,
 'window': 30,
 'val_frac': 0.2,
 'use_op_settings': False,
 'hidden_size': 64,
 'num_layers': 2,
 'dropout': 0.3,
 'lr': 0.001,
 'weight_decay': 1e-05,
 'batch_size': 256,
 'epochs': 100,
 'patience': 12,
 'seed': 42,
 'device': 'auto'}

## 1. Train
Set `RETRAIN=False` to skip training and only load the saved model (training takes ~4 min on one CPU core).

In [2]:
RETRAIN = True
if RETRAIN:
    out = train(cfg)
    history, d = out["history"], out["data"]
else:
    from src.data import prepare
    history = json.loads((ARTIFACT_DIR / "history.json").read_text()); d = prepare(cfg)
plots.fig_history(history); plt.show()

device=cpu | features=15 | train windows=14459 | val windows=3272 | test engines=100
constant columns dropped: ['op3', 's1', 's5', 's10', 's16', 's18', 's19']
features used: ['s2', 's3', 's4', 's6', 's7', 's8', 's9', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21']


epoch   1 | train RMSE  55.69 | val RMSE  31.04 *


epoch   2 | train RMSE  24.54 | val RMSE  18.66 *


epoch   3 | train RMSE  20.40 | val RMSE  16.23 *


epoch   4 | train RMSE  18.79 | val RMSE  15.01 *


epoch   5 | train RMSE  18.13 | val RMSE  15.51


... (9 outputs left out)

epoch  45 | train RMSE  13.96 | val RMSE  13.23 *


epoch  50 | train RMSE  13.91 | val RMSE  13.41


epoch  55 | train RMSE  13.86 | val RMSE  13.48


early stopping at epoch 57 (best val RMSE 13.229)
saved model + scaler + metadata to /home/claude/cmapss-rul/artifacts


## 2. Predict test RUL from each engine's **last 30 cycles**

In [3]:
from src.predict import load_predictor
pr = load_predictor()
pred = predict_cycles(pr.model, d["X_test"], cfg.rul_cap, pr.device)
true_raw = d["rul_true"].astype(float)
true_cap = np.minimum(true_raw, cfg.rul_cap)
res = pd.DataFrame({"vs official RUL_FD001": metrics(pred, true_raw),
                    "vs capped RUL (<=125)": metrics(pred, true_cap)}).T
res.round(2)

,RMSE,MAE,NASA_score
vs official RUL_FD001,14.65,11.14,362.36
vs capped RUL (<=125),13.13,10.07,261.03


In [4]:
plots.fig_pred_vs_true(pred, true_raw); plt.show()
plots.fig_error_hist(pred, true_raw); plt.show()
plots.fig_sorted(pred, true_raw); plt.show()

## 3. Rolling prediction for one engine
How the estimate evolves as more cycles are observed (this is what the Streamlit app shows live).

In [5]:
u = int(d["test_units"][np.argsort(true_raw)[50]])
g = d["test"][d["test"].unit == u]
roll = pr.predict_rolling(g)
roll["true_rul"] = np.minimum(int(true_raw[list(d["test_units"]).index(u)]) + (g.cycle.max() - roll.cycle), 125)
plots.fig_rolling(roll, unit=u); plt.show()

## 4. Error analysis
Engines close to failure matter most; large *positive* errors (late predictions) are the dangerous ones, which is exactly what the NASA score penalises.

In [6]:
df = pd.DataFrame({"unit": d["test_units"], "true": true_raw, "pred": pred.round(1)})
df["error"] = df.pred - df.true
print("RMSE when true RUL <= 30 :", round(float(np.sqrt((df[df.true<=30].error**2).mean())), 2))
print("RMSE when true RUL 30-80 :", round(float(np.sqrt((df[(df.true>30)&(df.true<=80)].error**2).mean())), 2))
print("RMSE when true RUL > 80  :", round(float(np.sqrt((df[df.true>80].error**2).mean())), 2))
df.reindex(df.error.abs().sort_values(ascending=False).index).head(8)

RMSE when true RUL <= 30 : 3.73
RMSE when true RUL 30-80 : 17.39
RMSE when true RUL > 80  : 16.55


,unit,true,pred,error
66,67,77.0,114.599998,37.599998
92,93,85.0,47.500000,-37.500000
24,25,145.0,108.199997,-36.800003
95,96,137.0,102.800003,-34.199997
44,45,114.0,81.199997,-32.800003
78,79,63.0,92.599998,29.599998
20,21,57.0,85.599998,28.599998
85,86,89.0,117.400002,28.400002
